# 24 aprile 2019 — reazione oraria alla nube di polvere

Come reagisce il modello **ora per ora** mentre la polvere arriva e se ne va.

Due scelte metodologiche, senza le quali il grafico misurerebbe altro:

**Le fasce sono definite sulla produzione attesa, non su quella realizzata.**
La produzione realizzata è l'esito: durante l'evento un nodo scivola nella
fascia bassa *proprio perché* la polvere l'ha colpito, quindi una linea per la
fascia alta descriverebbe i nodi risparmiati invece della reazione del modello.
L'atteso è la produzione dello stesso nodo alla stessa ora nei giorni tranquilli
precedenti: l'appartenenza alla fascia resta indipendente dall'evento.

**L'errore è relativo all'atteso e con segno.** Relativo perché l'errore in watt
cresce col sole e altrimenti si leggerebbe soprattutto l'alba; con segno perché
il punto è che la previsione è troppo alta all'arrivo e troppo bassa
all'uscita — l'errore assoluto cancellerebbe esattamente ciò che si vuole vedere.

Limite dichiarato: i nodi vengono colpiti in ore diverse, quindi sull'asse del
calendario il passaggio da sovrastima a sottostima appare più graduale di quanto
sia sul singolo nodo. La figura complementare, allineata sull'ora di arrivo di
ciascun nodo, sta nel notebook degli eventi estremi.

In [ ]:
from pathlib import Path
import importlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'physiq_pv').is_dir():
    raise FileNotFoundError('Avviare il notebook dalla root del repository o da notebooks/.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import physiq_pv.reporting.event_onset as event_onset
import physiq_pv.reporting.hourly_event_response as hourly_event_response
import physiq_pv.reporting.report_dump as report_dump

event_onset = importlib.reload(event_onset)
hourly_event_response = importlib.reload(hourly_event_response)
report_dump = importlib.reload(report_dump)

RUN_NAME = 'pvgis_stgnn_paper_faithful_gaussian_detector_mtgflow_ep60_seed1'
out_dir = ROOT / 'outputs' / RUN_NAME
if not (out_dir / 'predictions.csv').is_file():
    raise FileNotFoundError(out_dir / 'predictions.csv')
print('Run:', out_dir)

## 1. Finestra

I giorni dell'evento e i giorni tranquilli che costruiscono il riferimento. Il
21 aprile è escluso: la frazione regionale di nodi anomali sale già quel giorno,
quindi tenerlo nella baseline abbasserebbe l'atteso e nasconderebbe il calo.

In [ ]:
EVENT_DAYS = ['2019-04-22', '2019-04-23', '2019-04-24',
              '2019-04-25', '2019-04-26', '2019-04-27']
EXCLUDE_DAYS = ['2019-04-21']
BASELINE_DAYS = 14

window = event_onset.load_event_window(
    out_dir, event_days=EVENT_DAYS, baseline_days=BASELINE_DAYS,
    exclude_days=EXCLUDE_DAYS,
)
giorni = window.groupby(window['day'].dt.strftime('%Y-%m-%d'))['is_event'].first()
print('Giorni tranquilli:', list(giorni[~giorni].index))
print('Giorni evento    :', list(giorni[giorni].index))
print(f'Righe caricate   : {len(window):,}')

## 2. Profilo orario

`rel_bias` è `(previsto - reale) / atteso`: positivo significa che il modello
prevede più di quanto il nodo abbia prodotto, in proporzione a quanto avrebbe
dovuto produrre. `rel_true` e `rel_pred` sono gli stessi due livelli espressi
come quota dell'atteso, quindi `rel_true = 0.3` vuol dire che il nodo ha reso il
30% di una giornata normale a quell'ora.

In [ ]:
response = hourly_event_response.build_hourly_response(window)
profile = response['profile']

print('Fasce presenti:', response['bands'])
print('Nodi           :', response['n_nodes'])
display(profile.head(12))

# Ore dell'evento, fascia attesa piu alta: e' li che l'effetto e' piu grande.
top_band = response['bands'][-1]
event_rows = profile[(profile['band'] == top_band) & profile['is_event']]
display(
    event_rows[['timestamp', 'n', 'rel_true', 'rel_pred', 'rel_bias_median',
                'rel_bias_q1', 'rel_bias_q3', 'picp']]
    .set_index('timestamp').round(3)
)

## 3. Figure

Un pannello per fascia attesa: linea continua = mediana del bias relativo,
punteggiata = media, banda scura = Q1–Q3 fra i nodi, banda chiara = P10–P90. I
cerchietti vuoti marcano le ore calcolate su meno di `MIN_N` nodi, dove il
valore oscilla per numerosità e non per comportamento del modello. L'ultimo
pannello riporta `n`.

In [ ]:
MIN_N = 20
figure_dir = out_dir / 'figures' / 'april_dust_hourly'
figure_dir.mkdir(parents=True, exist_ok=True)

fig = hourly_event_response.plot_hourly_response(
    response, event_days=EVENT_DAYS, min_n=MIN_N,
    title='24 aprile 2019 — bias relativo orario per fascia di produzione attesa',
)
path = figure_dir / 'hourly_relative_bias.png'
fig.savefig(path, dpi=140, bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(path)))

In [ ]:
# Livelli: quanto ha prodotto e quanto ha previsto, in quota dell'atteso.
for band in response['bands']:
    fig = hourly_event_response.plot_hourly_levels(
        response, band=band, event_days=EVENT_DAYS,
        title=f'Livelli orari — fascia {band}',
    )
    path = figure_dir / f'hourly_levels_{band}.png'
    fig.savefig(path, dpi=140, bbox_inches='tight')
    plt.close(fig)
    print(path.name)
    display(Image(filename=str(path)))

## 4. Quando cambia segno

Il ritardo di reazione si legge come l'intervallo fra la prima ora di
sovrastima e il passaggio a sottostima, fascia per fascia.

In [ ]:
rows = []
for band in response['bands']:
    block = profile[(profile['band'] == band) & profile['is_event']].sort_values('timestamp')
    block = block[block['n'] >= MIN_N]
    if block.empty:
        continue
    over = block[block['rel_bias_median'] > 0.05]
    under = block[block['rel_bias_median'] < -0.05]
    rows.append({
        'fascia': band,
        'ore_sovrastima': len(over),
        'ore_sottostima': len(under),
        'prima_sovrastima': over['timestamp'].min() if not over.empty else None,
        'picco_sovrastima': over['rel_bias_median'].max() if not over.empty else None,
        'prima_sottostima': under['timestamp'].min() if not under.empty else None,
        'picco_sottostima': under['rel_bias_median'].min() if not under.empty else None,
        'picp_min': block['picp'].min() if 'picp' in block else None,
    })
crossing = pd.DataFrame(rows)
display(crossing)

## 5. Riepilogo da copiare

In [ ]:
report_dump.dump_sections({
    'profilo_orario_fascia_alta': event_rows[
        ['timestamp', 'band', 'n', 'rel_true', 'rel_pred', 'rel_bias_median',
         'rel_bias_q1', 'rel_bias_q3', 'picp']
    ],
    'cambi_di_segno': crossing if 'crossing' in dir() else None,
}, path=out_dir / 'analysis_summary_april_hourly.txt', max_rows=80)